<a href="https://colab.research.google.com/github/Ehsan-Roohi/Introduction-to-Compressible-Flows/blob/main/notebooks/chapter08/08_normal_shock_location_cd_nozzle.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Normal-Shock Location in a Converging-Diverging Nozzle

**Book:** *Introduction to Compressible Flows*<br>
**Chapter 8:** Quasi-One-Dimensional Flow in Converging-Diverging Nozzles<br>
**Concepts:** area-Mach relation · nozzle choking · normal-shock total-pressure loss · shock location

## Learning objectives

- Check whether the nozzle is choked for the prescribed back pressure.
- Infer the subsonic exit Mach number and total-pressure loss.
- Invert the normal-shock relation and report the shock area ratio As/At.

## How to use this notebook

Edit the reservoir pressure, back pressure, exit-to-throat area ratio, and gamma in the worked example and run the cell.

> **Model scope.** The result locates the shock by area ratio in a quasi-one-dimensional, adiabatic, perfect-gas nozzle; converting As/At to an axial coordinate requires the nozzle geometry A(x).


In [ ]:
import math

# ----------------------------
# Isentropic relations
# ----------------------------

def critical_pressure_ratio(gamma: float) -> float:
    """
    Critical pressure ratio for choking:
    (Pb/Pt)_critical = (2/(gamma+1))^(gamma/(gamma-1))
    """
    return (2.0 / (gamma + 1.0)) ** (gamma / (gamma - 1.0))

def p0_over_p(M: float, gamma: float) -> float:
    """
    Isentropic stagnation-to-static pressure ratio:
    P0/P = (1 + (gamma-1)/2 * M^2)^(gamma/(gamma-1))
    """
    return (1.0 + 0.5 * (gamma - 1.0) * M * M) ** (gamma / (gamma - 1.0))

def area_over_astar(M: float, gamma: float) -> float:
    """
    Isentropic area-Mach relation:
    A/A* as a function of Mach number
    """
    g = gamma
    return (1.0 / M) * (
        (2.0 / (g + 1.0) * (1.0 + 0.5 * (g - 1.0) * M * M))
        ** ((g + 1.0) / (2.0 * (g - 1.0)))
    )

# ----------------------------
# Normal shock relations
# ----------------------------

def normal_shock_M2(M1: float, gamma: float) -> float:
    """
    Downstream Mach number after a normal shock
    """
    g = gamma
    return math.sqrt(
        (1.0 + 0.5 * (g - 1.0) * M1 * M1) /
        (g * M1 * M1 - 0.5 * (g - 1.0))
    )

def p2_over_p1(M1: float, gamma: float) -> float:
    """
    Static pressure ratio across a normal shock (P2/P1)
    """
    g = gamma
    return 1.0 + (2.0 * g / (g + 1.0)) * (M1 * M1 - 1.0)

def p02_over_p01(M1: float, gamma: float) -> float:
    """
    Total pressure ratio across a normal shock (P02/P01)

    P02/P01 = (P2/P1) * [(P02/P2) / (P01/P1)]
    """
    M2 = normal_shock_M2(M1, gamma)
    return p2_over_p1(M1, gamma) * (
        p0_over_p(M2, gamma) / p0_over_p(M1, gamma)
    )

# ----------------------------
# Chapter 8 explicit exit-Mach relation
# Explicit formula for exit Mach number Me
# ----------------------------

def Me_from_equation_7_85(X: float, gamma: float) -> float:
    """
    Chapter 8 explicit exit-Mach relation:

    Me^2 = -1/(gamma-1)
           + sqrt[ 1/(gamma-1)^2
                   + (2/(gamma-1)) *
                     (2/(gamma+1))^((gamma+1)/(gamma-1)) *
                     X^2 ]

    where:
        X = (p0e * Ae*) / (pe * Ae)

    Returns subsonic Me.
    """
    g = gamma
    if X <= 0.0:
        raise ValueError("X must be positive.")

    a = g - 1.0

    term_inside = (
        1.0 / (a * a)
        + (2.0 / a)
        * (2.0 / (g + 1.0)) ** ((g + 1.0) / a)
        * (X * X)
    )

    Me2 = (-1.0 / a) + math.sqrt(term_inside)

    if Me2 <= 0.0:
        raise ValueError("Computed Me^2 is non-positive.")

    Me = math.sqrt(Me2)

    if Me >= 1.0:
        raise ValueError("Computed Me is not subsonic.")

    return Me

# ----------------------------
# Invert normal shock table (numerical step)
# ----------------------------

def M1_from_p02p01(target_ratio: float, gamma: float) -> float:
    """
    Find upstream supersonic Mach number M1
    such that p02_over_p01(M1) = target_ratio.

    This replaces the use of the normal shock table.
    """
    if not (0.0 < target_ratio < 1.0):
        raise ValueError("Target p02/p01 must be between 0 and 1.")

    lo, hi = 1.0000001, 50.0

    for _ in range(300):
        mid = 0.5 * (lo + hi)
        fmid = p02_over_p01(mid, gamma) - target_ratio

        if abs(fmid) < 1e-12:
            return mid

        # For M1 > 1, p02/p01 decreases as M1 increases
        if fmid > 0:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

# ----------------------------
# Full textbook procedure
# ----------------------------

def solve_shock_location_textbook(P01: float, pe: float, Ae_At: float, gamma: float):
    """
    Textbook method (step-by-step):

    Step 1:
        R = (pe / P01) * (Ae / At)

        X = (p0e * Ae*) / (pe * Ae) = 1 / R

    Step 2:
        Compute exit Mach number Me using Equation (7-85)

    Step 3:
        Compute p0e/pe from isentropic relation
        Then compute p02/p01:
            p02/p01 = (p0e/pe) * (pe/P01)

    Step 4:
        Use normal shock relation (table inversion)
        to find upstream Mach number M1

    Step 5:
        Determine shock location:
            As/At = (A/A*)(M1)

    Returns:
        (As/At, M1, M2, Me, p02/p01)
    """

    # Step 1
    R = (pe / P01) * Ae_At
    if R <= 0.0:
        raise ValueError("Invalid pressure/area combination.")

    X = 1.0 / R

    # Step 2
    Me = Me_from_equation_7_85(X, gamma)

    # Step 3
    p0e_over_pe = p0_over_p(Me, gamma)
    p02_over_p01_value = p0e_over_pe * (pe / P01)

    # Step 4
    M1 = M1_from_p02p01(p02_over_p01_value, gamma)
    M2 = normal_shock_M2(M1, gamma)

    # Step 5
    As_At = area_over_astar(M1, gamma)

    return As_At, M1, M2, Me, p02_over_p01_value

# ----------------------------
# MAIN
# ----------------------------





# Reproducible worked example
P01 = 500_000.0       # Pa
pe = 150_000.0        # Pa
Ae_At = 2.5
gamma = 1.4

if pe / P01 >= critical_pressure_ratio(gamma):
    print("Flow is not choked; no internal normal-shock solution is reported.")
else:
    As_At, M1, M2, Me, p02p01 = solve_shock_location_textbook(P01, pe, Ae_At, gamma)
    print(f"exit Mach Me = {Me:.6f}")
    print(f"pre-/post-shock Mach = {M1:.6f} / {M2:.6f}")
    print(f"total-pressure ratio p02/p01 = {p02p01:.6f}")
    print(f"shock location As/At = {As_At:.6f}")


## Interpretation and verification checklist

- An internal normal shock is considered only after the choking test passes.
- The pre-shock Mach number must be supersonic and the post-shock Mach number subsonic.
- Total pressure must decrease across the shock, so p02/p01 lies between zero and one.

## Reproducibility note

Restart the runtime and run the notebook from top to bottom after changing inputs. Record the input values, units, heat-capacity ratio, numerical tolerances, and dependency versions with any result used in coursework, research, or a publication.

Questions or reproducibility reports are welcome through the repository's issue templates.
